# Multiple dispatch

**Multiple dispatch** is a key feature of Julia, that we will explore in this notebook.

It helps make software fast. It also makes software extensible, programmable, and downright fun to play with. 

It may just herald a breakthrough for parallel computation.

1. Roman numerals
2. Functions
3. Parallel computing

## 1. Roman numerals (for fun)

Let's define a **new struct** that represents a Roman numeral. For coding simplicity, we'll just deal with numbers between 0 and 9. 

**Exercise**: Extend this to larger numbers. (Recall that Roman numbers are a base-10 system!)

In [2]:
struct Roman
    n::Int
end

Base.show(io::IO, r::Roman) = print(io, 'ⅰ' + (r.n - 1) % 10)  # nice display; 'ⅰ' is a Unicode Roman numeral

We can create an object of this type as follows:

In [16]:
Roman(4)

ⅳ

In [4]:
typeof.([5 5.0 Roman(5) "Five" '5'  5//1])

1×6 Matrix{DataType}:
 Int64  Float64  Roman  String  Char  Rational{Int64}

We would like to display it nicely, in Roman numerals:

In [5]:
x = [7 1 2 5 8 9]
Roman.(x)   # equivalent to map(Roman, x)  or  [Roman(w) for w in x]

1×6 Matrix{Roman}:
 ⅶ  ⅰ  ⅱ  ⅴ  ⅷ  ⅸ

It'd be nice to be able to add Roman numerals together like normal numbers:

In [6]:
Roman(4) + Roman(5)

MethodError: MethodError: no method matching +(::Roman, ::Roman)
The function `+` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  +(::Any, ::Any, !Matched::Any, !Matched::Any...)
   @ Base operators.jl:642
  +(!Matched::Pkg.Resolve.FieldValue, !Matched::Pkg.Resolve.FieldValue)
   @ Pkg /opt/homebrew/Cellar/julia/1.12.7/share/julia/stdlib/v1.12/Pkg/src/Resolve/fieldvalues.jl:45
  +(!Matched::Bool, !Matched::Complex{Bool})
   @ Base complex.jl:308
  ...


But Julia doesn't know how to do that. Let's teach it by `import`ing the `+` function, which then allows us to _extend_ its definition:

In [7]:
import Base: +, *

+(a::Roman, b::Roman) = Roman(a.n + b.n)

+ (generic function with 192 methods)

In [9]:
Roman(4) + Roman(5)

ⅸ

This **adds a new method** to the function `+`:

In [10]:
methods(+)

# 192 methods for generic function "+" from Base:
   [1] +(a::Pkg.Resolve.FieldValue, b::Pkg.Resolve.FieldValue)
     @ /opt/homebrew/Cellar/julia/1.12.7/share/julia/stdlib/v1.12/Pkg/src/Resolve/fieldvalues.jl:45
   [2] +(x::Bool, z::Complex{Bool})
     @ complex.jl:308
   [3] +(x::Bool, y::Bool)
     @ bool.jl:168
   [4] +(x::Bool)
     @ bool.jl:165
   [5] +(x::Bool, z::Complex)
     @ complex.jl:315
   [6] +(x::Bool, y::T) where T<:AbstractFloat
     @ bool.jl:175
   [7] +(x::BigInt, y::BigInt)
     @ gmp.jl:502
   [8] +(a::BigInt, b::BigInt, c::BigInt)
     @ gmp.jl:542
   [9] +(a::BigInt, b::BigInt, c::BigInt, d::BigInt)
     @ gmp.jl:543
  [10] +(a::BigInt, b::BigInt, c::BigInt, d::BigInt, e::BigInt)
     @ gmp.jl:544
  [11] +(x::BigInt, y::BigInt, rest::BigInt...)
     @ gmp.jl:679
  [12] +(c::BigInt, x::BigFloat)
     @ mpfr.jl:613
  [13] +(x::BigInt, c::Union{UInt16, UInt32, UInt64, UInt8})
     @ gmp.jl:550
  [14] +(x::BigInt, c::Union{Int16, Int32, Int64, Int8})
     @ gmp.jl:556
  [15] +(::Missing, ::Missing)
     @ missing.jl:122
  [16] +(::Missing)
     @ missing.jl:101
  [17] +(x::Missing, y::Dates.AbstractTime)
     @ /opt/homebrew/Cellar/julia/1.12.7/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:91
  [18] +(::Missing, ::Number)
     @ missing.jl:123
  [19] +(x::Rational{BigInt}, y::Rational{BigInt})
     @ gmp.jl:1062
  [20] +(t::Dates.Time, dt::Dates.Date)
     @ /opt/homebrew/Cellar/julia/1.12.7/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:22
  [21] +(x::Dates.Time, y::Dates.TimePeriod)
     @ /opt/homebrew/Cellar/julia/1.12.7/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:85
  [22] +(B::BitMatrix, J::LinearAlgebra.UniformScaling)
     @ /opt/homebrew/Cellar/julia/1.12.7/share/julia/stdlib/v1.12/LinearAlgebra/src/uniformscaling.jl:154
  [23] +(a::Pkg.Resolve.VersionWeight, b::Pkg.Resolve.VersionWeight)
     @ /opt/homebrew/Cellar/julia/1.12.7/share/julia/stdlib/v1.12/Pkg/src/Resolve/versionweights.jl:22
  [24] +(z::Complex{Bool}, x::Bool)
     @ complex.jl:309
  [25] +(z::Complex{Bool}, x::Real)
     @ complex.jl:323
  [26] +(dt::Dates.Date, t::Dates.Time)
     @ /opt/homebrew/Cellar/julia/1.12.7/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:21
  [27] +(dt::Dates.Date, y::Dates.Year)
     @ /opt/homebrew/Cellar/julia/1.12.7/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:29
  [28] +(dt::Dates.Date, z::Dates.Month)
     @ /opt/homebrew/Cellar/julia/1.12.7/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:56
  [29] +(x::Dates.Date, y::Dates.Quarter)
     @ /opt/homebrew/Cellar/julia/1.12.7/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:75
  [30] +(x::Dates.Date, y::Dates.Week)
     @ /opt/homebrew/Cellar/julia/1.12.7/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:79
  [31] +(x::Dates.Date, y::Dates.Day)
     @ /opt/homebrew/Cellar/julia/1.12.7/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:81
  [32] +(a::Roman, b::Roman)
     @ ~/Documents/MP/JuliaTutorials/introductory-tutorials/intro-to-julia/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X20sZmlsZQ==.jl:3
  [33] +(x::Dates.CompoundPeriod, y::Dates.CompoundPeriod)
     @ /opt/homebrew/Cellar/julia/1.12.7/share/julia/stdlib/v1.12/Dates/src/periods.jl:335
  [34] +(x::Dates.CompoundPeriod, y::Dates.Period)
     @ /opt/homebrew/Cellar/julia/1.12.7/share/julia/stdlib/v1.12/Dates/src/periods.jl:333
  [35] +(x::Dates.CompoundPeriod, y::Dates.TimeType)
     @ /opt/homebrew/Cellar/julia/1.12.7/share/julia/stdlib/v1.12/Dates/src/periods.jl:363
  [36] +(level::Base.CoreLogging.LogLevel, inc::Integer)
     @ logging/logging.jl:134
  [37] +(dt::Dates.DateTime, y::Dates.Year)
     @ /opt/homebrew/Cellar/julia/1.12.7/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:25
  [38] +(dt::Dates.DateTime, z::Dates.Month)
     @ /opt/homebrew/Cellar/julia/1.12.7/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:49
  [39] +(x::Dates.DateTime, y::Dates.Quarter)
     @ /opt/homebrew/Cellar/julia/1.12.7/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:77
  [40] +(x::

In [11]:
import Base.*
*(i::Roman, j::Roman) = Roman(i.n * j.n)                     # Multiply like a Roman

* (generic function with 189 methods)

In [12]:
Roman(3) * Roman(2)

ⅵ

In [13]:
Roman.(1:3) .* [Roman(1) Roman(2) Roman(3)]

3×3 Matrix{Roman}:
 ⅰ  ⅱ  ⅲ
 ⅱ  ⅳ  ⅵ
 ⅲ  ⅵ  ⅸ

But 

In [ ]:
Roman(3) * 2

ⅵ

In [17]:
# Complicated mytimes to decide what to do based on type
# not suggested, better way coming soon
function mytimes(i,j)
  if isa(i,Roman) & isa(j,Number)
        return  fill(1, i.n, j)   # i by j matrix with ones
    elseif    isa(i,Number) & isa(j,Roman) 
        return "😄"^(i*j.n)   #  i * j happy faces
    else
        return("I Don't know")
    end
end

mytimes (generic function with 1 method)

In [18]:
mytimes(4,Roman(3)) # Twelve happys

"😄😄😄😄😄😄😄😄😄😄😄😄"

In [19]:
mytimes(Roman(4),3) # 4x3 matrix with ones

4×3 Matrix{Int64}:
 1  1  1
 1  1  1
 1  1  1
 1  1  1

The simplest thing to do is to explicitly define multiplication of a `Roman` by a number. We can do it as we see fit:

In [20]:
*(i::Number, j::Roman) = "😄"^(i*j.n)        #  i * j happy faces

*(i::Roman, j::Number) = fill(1, i.n, j)       # i by j matrix

* (generic function with 191 methods)

In [21]:
3 * Roman(3) # Nine happys

"😄😄😄😄😄😄😄😄😄"

In [22]:
Roman(3) * 5  # Three by Five matrix of ones

3×5 Matrix{Int64}:
 1  1  1  1  1
 1  1  1  1  1
 1  1  1  1  1

In [23]:
t(x::Roman,y::Roman) = x.n * y.n

t (generic function with 1 method)

In [24]:
t(Roman(5),Roman(4))

20

In [25]:
# Notice how tight the assembler is!
@code_native t(Roman(2),Roman(4))

	.section	__TEXT,__text,regular,pure_instructions
	.build_version macos, 16, 0
	.globl	_julia_t_6786                   ; -- Begin function julia_t_6786
	.p2align	2
_julia_t_6786:                          ; @julia_t_6786
; Function Signature: t(Main.Roman, Main.Roman)
; ┌ @ /Users/maevegualtieri-reed/Documents/MP/JuliaTutorials/introductory-tutorials/intro-to-julia/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X42sZmlsZQ==.jl:1 within `t`
; %bb.0:                                ; %top
	;DEBUG_VALUE: t:x <- [$x0+0]
	;DEBUG_VALUE: t:x <- [$x0+0]
	;DEBUG_VALUE: t:y <- [$x1+0]
	;DEBUG_VALUE: t:y <- [$x1+0]
; │┌ @ int.jl:88 within `*`
	ldr	x8, [x0]
	ldr	x9, [x1]
	mul	x0, x9, x8
	ret
; └└
                                        ; -- End function
.subsections_via_symbols


## Functions

In [26]:
import Base: *, +, ^

In [27]:
*(α::Number,   g::Function) = x -> α * g(x)   # Scalar times function

*(f::Function, λ::Number)   = x -> f(λ * x)   # Scale the argument

*(f::Function, g::Function) = x -> f(g(x))    # Function composition  -- abuse of notation!  use \circ in Julia 0.6

^(f::Function, n::Integer) = n == 1 ? f : f*f^(n-1) # A naive exponentiation algorithm by recursive multiplication

^ (generic function with 73 methods)

In [28]:
+(f::Function, g::Function) = x -> f(x) + g(x)

+ (generic function with 193 methods)

For example, the exponential function is defined as

$$\exp(x) = \sum_{n=0}^\infty \frac{1}{n!} x^n.$$

We can think of this just in terms of functions:

$$\exp = \sum_{n=0}^\infty \frac{1}{n!} \mathrm{pow}_n,$$

where $\mathrm{pow}_n(x) = x^n$.

(starts to blur the symbolic with the numerical!)

In [29]:
pow(n) = x -> x^n

myexp = x -> sum(1/factorial(big(n)) * pow(n)(x) for n in 0:100)   # taylor series not efficient!

#29 (generic function with 1 method)

In [30]:
[myexp(1); exp(1); exp(big(1))]

3-element Vector{BigFloat}:
 2.718281828459045235360287471352662497757247093699959574966967627724076630353416
 2.718281828459045090795598298427648842334747314453125
 2.718281828459045235360287471352662497757247093699959574966967627724076630353555

In [31]:
f = x -> x^2
f(10)

100

In [32]:
g = 3f
g(10)

300

In [33]:
(f^2)(10)  # since we defined multiplication of functions as composition

10000

In [34]:
using Plots;
gr()

Plots.GRBackend()

In [ ]:
x = pi*(0:0.001:4)

plot(x, sin.(x),    c="black", label="Fun")
plot!(x, (12*sin).(x),    c="green", label="Num * Fun")
plot!(x, (sin*12).(x),    c="red", alpha=0.9, label="Fun * Num")
plot!(x, (5*sin*exp).(x), c="blue", alpha=0.2, label="Num * Fun * Fun")

In [ ]:
plot([12*sin, sin*12, 5*sin*exp], 0:.01:4π, α=[1 .9 .2], c=[:green :red :blue])

<img src="https://lh4.googleusercontent.com/--z5eKJbB7sg/UffjL1iAd4I/AAAAAAAABOc/S_wDVyDOBfQ/gauss.jpg">

###  "Sin^2 phi is odious to me, even though Laplace made use of it; should  it be feared that sin^2 phi might become ambiguous, which would perhaps  never occur, or at most very rarely when speaking of sin(phi^2), well  then, let us write (sin phi)^2, but not sin^2 phi, which by analogy  should signify sin(sin phi)." -- Gauss

In [ ]:
x=(0:.01:2) * pi;

plot(x, (sin^2).(x), c="blue")     # Squaring just works, y=sin(sin(x)), Gauss would be pleased!
plot!(x, sin.(x).^2,  c="red")         

# Exercise

In [ ]:
h(a, b::Any) = "fallback"
h(a::Number, b::Number) = "a and b are both numbers"
h(a::Number, b) = "a is a number"
h(a, b::Number) = "b is a number"
h(a::Integer, b::Integer) = "a and b are both integers"

In [ ]:
# Try playing with h